In [1]:
import pandas as pd
import pickle
import time

from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv(r"C:\Users\ELDRICK VICTORIA\Desktop\toxic-XAI\version_2\data\processed\cleaned_data.csv")

df = df[["clean_text", "target"]].dropna()

df = df.sample(n=20000, random_state=42)

X = df["clean_text"]
y = df["target"]

print(df.shape)

(20000, 2)


In [3]:
v1_models = {
    "Logistic_V1": pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_1\\models\\logistic_regression.pkl", "rb")),
    "SVM_V1": pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_1\\models\\linear_svm.pkl", "rb")),
    "RF_V1": pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_1\\models\\random_forest.pkl", "rb")),
}

v1_vectorizer = pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_1\\models\\tfidf.pkl", "rb"))

c:\Users\ELDRICK VICTORIA\Desktop\toxic-XAI\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ELDRICK VICTORIA\Desktop\toxic-XAI\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LinearSVC from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ELDRICK VICTORIA\Desktop\toxic-XAI\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeC

In [4]:
v2_models = {
    "Logistic_V2": pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_2\\models\\logistic_regression.pkl", "rb")),
    "SVM_V2": pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_2\\models\\linear_svm.pkl", "rb")),
}

v2_vectorizer = pickle.load(open("C:\\Users\\ELDRICK VICTORIA\\Desktop\\toxic-XAI\\version_2\\models\\tfidf_vectorizer.pkl", "rb"))

In [5]:
def evaluate(models, vectorizer, X, y):

    results = []

    X_vec = vectorizer.transform(X)

    for name, model in models.items():

        start = time.time()

        y_pred = model.predict(X_vec)

        end = time.time()

        acc = accuracy_score(y, y_pred)

        report = classification_report(y, y_pred, output_dict=True)

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision_Toxic": report["1"]["precision"],
            "Recall_Toxic": report["1"]["recall"],
            "F1_Toxic": report["1"]["f1-score"],
            "Time": end - start
        })

    return pd.DataFrame(results)

In [6]:
v1_results = evaluate(v1_models, v1_vectorizer, X, y)
v2_results = evaluate(v2_models, v2_vectorizer, X, y)

v1_results, v2_results

(         Model  Accuracy  Precision_Toxic  Recall_Toxic  F1_Toxic      Time
 0  Logistic_V1   0.93170         0.880682      0.263009  0.405052  0.001000
 1       SVM_V1   0.93435         0.747552      0.388575  0.511351  0.001009
 2        RF_V1   0.91170         1.000000      0.001131  0.002260  0.032129,
          Model  Accuracy  Precision_Toxic  Recall_Toxic  F1_Toxic      Time
 0  Logistic_V2   0.88940         0.417960      0.639706  0.505588  0.000991
 1       SVM_V2   0.88855         0.412257      0.612557  0.492833  0.000523)

In [7]:
final_results = pd.concat([v1_results, v2_results], ignore_index=True)

final_results.sort_values(by="F1_Toxic", ascending=False)

,Model,Accuracy,Precision_Toxic,Recall_Toxic,F1_Toxic,Time
1,SVM_V1,0.93435,0.747552,0.388575,0.511351,0.001009
3,Logistic_V2,0.88940,0.417960,0.639706,0.505588,0.000991
4,SVM_V2,0.88855,0.412257,0.612557,0.492833,0.000523
0,Logistic_V1,0.93170,0.880682,0.263009,0.405052,0.001000
2,RF_V1,0.91170,1.000000,0.001131,0.002260,0.032129


In [8]:
bert_result = pd.DataFrame([{
    "Model": "BERT_V2",
    "Accuracy": 0.98155,
    "Precision_Toxic": 0.90,
    "Recall_Toxic": 0.89,
    "F1_Toxic": 0.90,
    "Time": 235.16
}])

final_results = pd.concat([final_results, bert_result], ignore_index=True)

final_results.sort_values(by="F1_Toxic", ascending=False)

,Model,Accuracy,Precision_Toxic,Recall_Toxic,F1_Toxic,Time
5,BERT_V2,0.98155,0.900000,0.890000,0.900000,235.160000
1,SVM_V1,0.93435,0.747552,0.388575,0.511351,0.001009
3,Logistic_V2,0.88940,0.417960,0.639706,0.505588,0.000991
4,SVM_V2,0.88855,0.412257,0.612557,0.492833,0.000523
0,Logistic_V1,0.93170,0.880682,0.263009,0.405052,0.001000
2,RF_V1,0.91170,1.000000,0.001131,0.002260,0.032129


# 🧠 Final Analysis Report — Toxic Comment Classification (Version 1 vs Version 2)

---

## 🎯 Overview

This project evaluates multiple machine learning and deep learning models for **toxic comment classification**, comparing two versions of the pipeline:

* **Version 1:** Traditional ML models with basic preprocessing
* **Version 2:** Improved preprocessing + optimized ML + BERT

The evaluation focuses on **real-world performance**, especially on the **toxic class**, which is a minority and harder to detect.

---

## 📊 Evaluation Metrics

The models were evaluated using:

* Accuracy
* Precision (Toxic Class)
* Recall (Toxic Class)
* **F1-score (Toxic Class)** ⭐ *(Primary metric due to class imbalance)*
* Inference Time

---

## 🏆 Best Performing Model

**BERT_V2** is the best model across all metrics:

* Accuracy: **98.15%**
* Precision (Toxic): **0.90**
* Recall (Toxic): **0.89**
* F1-score (Toxic): **0.90**

### ✅ Why BERT Performs Best

* Captures **contextual meaning** instead of just keywords
* Handles **sarcasm, implicit toxicity, and complex language**
* Achieves **balanced precision and recall**
* Significantly improves **minority class detection**

---

## 📊 Detailed Model Comparison

### 🔹 Classical Models (Version 1 & Version 2)

#### SVM_V1

* Accuracy: 93.43%
* Precision: 0.75
* Recall: 0.38
* F1-score: 0.51

**Insight:**
Good precision but very low recall → misses many toxic comments.

---

#### Logistic_V2

* Accuracy: 88.94%
* Precision: 0.42
* Recall: 0.64
* F1-score: 0.51

**Insight:**
Higher recall but low precision → detects more toxic comments but with more false positives.

---

#### SVM_V2

* Accuracy: 88.85%
* Precision: 0.41
* Recall: 0.61
* F1-score: 0.49

**Insight:**
Balanced but not significantly better than Logistic Regression.

---

#### Logistic_V1

* Accuracy: 93.17%
* Precision: 0.88
* Recall: 0.26
* F1-score: 0.40

**Insight:**
Very high precision but extremely poor recall → fails to detect most toxic comments.

---

## 🚨 Random Forest (Failure Case)

* Precision: 1.00
* Recall: 0.001
* F1-score: 0.002

**Insight:**
Random Forest predicts almost all comments as **non-toxic**, resulting in:

* Near-zero recall
* Useless real-world performance

### 💡 Reason

Random Forest is **not suitable for high-dimensional sparse TF-IDF features**, where linear models perform better.

---

## 🔄 Version 1 vs Version 2

### Improvements in Version 2:

* Better text preprocessing
* Improved handling of noisy and imbalanced data
* Slight improvements in ML model recall
* Introduction of **BERT (major improvement)**

### Performance Change:

| Model       | F1 (Toxic) |
| ----------- | ---------- |
| Logistic_V1 | 0.40       |
| Logistic_V2 | 0.50       |
| SVM_V1      | 0.51       |
| SVM_V2      | 0.49       |
| **BERT_V2** | **0.90**   |

**Conclusion:**
Version 2 improves ML models slightly, but the **major performance leap comes from BERT**.

---

## ⚖️ Trade-Off Analysis

| Factor                | ML Models             | BERT            |
| --------------------- | --------------------- | --------------- |
| Speed                 | ⚡ Very Fast (~0.001s) | 🐢 Slow (~235s) |
| Accuracy              | Medium                | Very High       |
| Recall (Toxic)        | Low–Medium            | High            |
| Memory Usage          | Low                   | High            |
| Context Understanding | ❌                     | ✅               |

---

## 🧠 Key Insights

* High accuracy can be misleading in imbalanced datasets
* Classical models struggle with **context and semantics**
* Toxic detection requires understanding **intent, not just words**
* BERT significantly improves **minority class performance**

---

## 🎯 Final Conclusion

Traditional machine learning models such as Logistic Regression and SVM provide fast and efficient baseline solutions but struggle with detecting toxic comments due to poor contextual understanding and low recall for the minority class.

Version 2 improves upon Version 1 through better preprocessing and feature engineering, but the most significant improvement comes from introducing BERT.

BERT achieves a substantial increase in F1-score for the toxic class (0.90 compared to ~0.5 for classical models), making it far more suitable for real-world toxic comment moderation systems.

However, this performance gain comes at the cost of higher computational complexity and inference time, highlighting the trade-off between efficiency and accuracy.

---

## 💼 Key Takeaways

* Version 2 outperforms Version 1
* BERT is the best-performing model
* Random Forest is unsuitable for this problem
* Classical models serve as fast baselines
* BERT is recommended for production-level accuracy

---

## 🧠 One-Line Summary (Interview Ready)

BERT significantly outperforms traditional models by capturing contextual meaning, improving toxic comment detection F1-score from ~0.5 to 0.9, making it the most suitable model despite higher computational cost.

---
